# Activity 3: Mapping Fine-Grained Migration Flows

In this notebook, you will map some of our migration data.

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

We probably will not have time to go over both case studies in the workshop---feel free to skip #2 and go ahead to #3 if you are more interested in wildfires.

Keep this link open, it could be helpful if you are not familiar with Geopandas: https://geopandas.org/en/stable/docs/user_guide/mapping.html

You should add the csv you received with MIGRATE data to the `data` directory. Note that the file contains only flows from 2018 to 2019, departing either California or New York CBGs.

In [ ]:
MIGRATE_data = pd.read_csv('data/MIGRATE_sample_2018-2019.csv')

## 1. Collecting Census spatial data

There are several ways to collect Census spatial data. Here we will use [pygris](https://walker-data.com/pygris/). It needs to be installed via pip, so run the following cell:

In [ ]:
! pip install pygris

You should be able to now import pygris and use the `block_groups()` function to collect all CBGs in a given state. This will take ~1 minute. You may see some errors of HTTP download---the Census website is in flux these days, sadly, and pygris is trying to look for alternatives to download the data

In [ ]:
import pygris
NY_CBGs = pygris.block_groups(state='36', year=2010) #New York state
CA_CBGs = pygris.block_groups(state='06', year=2010) #California

This is how one such dataframe looks like:

In [ ]:
NY_CBGs.head()

## 2. Mapping flows in New York City

We will start mapping flows in New York City, showing one advantage of fine-grained migration data: it reveals urban, neighborhood-level dynamics.

First, we select the CBGs in New York City --- that is, CBGs in one of the 5 NYC counties: Manhattan, Queens, Bronx, Brooklyn, and Staten Island


In [ ]:
NYC_counties = {'Bronx': '005', 'Brooklyn':'047', 'Manhattan':'061', 'Queens':'081', 'Staten Island':'085'}
NYC_CBGs = NY_CBGs[NY_CBGs['COUNTYFP10'].isin(NYC_counties.values())].reset_index(drop=True)

To practice a simple plot:

In [ ]:
NYC_CBGs.plot() #Just call plot on a geodataframe!

You can treat geodataframe plots as any other matplotlib element. For example you could pass an ax or a figsize. See additional parameters:

In [ ]:
NYC_CBGs.plot?

Note that the water is part of the CBGs---this is not desired sometimes. We could solve the issue by downloading the boundary of NYC and cropping all CBGs (a spatial overlay, or clip---see the Geopandas docs!). We could also use `pygris`, which has an `erase_water` function!. Note that this function is a bit slow---it needs to download water data---and also it will change the index of the geodataframe! So be careful

In [ ]:
#This takes a while t run---40 seconds for me. It needs to download water data. Feel free to skip.
from pygris.utils import erase_water
NYC_CBGs_nowater = erase_water(NYC_CBGs)
NYC_CBGs_nowater.plot(figsize=(10,10))

### 1. Visualizing flows from Manhattan

Let's visualize flows from Manhattan, classified according to the destination. First, look at the MIGRATE dataframe. You will see that GEOIDs are given as integers---it is good practice to make sure they are strings (12 digits for CBGs).

In [ ]:
MIGRATE_data['Origin'] = MIGRATE_data['Origin'].astype(str).str.zfill(12)
MIGRATE_data['Destination'] = MIGRATE_data['Destination'].astype(str).str.zfill(12)
MIGRATE_data.head()

We will then need to select the CBGs in Manhattan. To do that, note that GEOIDs are built geography by geography: they begin with the 2-digit state code (36 for NY), then the 3 digit county code (which... we have defined above for Manhattan---can you find it?).

In [ ]:
start_of_a_Manhattan_GEOID = '36' + '...'#Manhattan FIPS code? what is it?
flows_out_of_Manhattan = MIGRATE_data.loc[MIGRATE_data['Origin'].str.startswith(start_of_a_Manhattan_GEOID)]

Now, we can use similar selections to find flows from Manhattan to different destinations. For the exercise, define NYC as the 5 counties above. That is someone moves outside of NYC if it doesnt move to Manhattan, Queens, Bronx, Brooklyn, or Staten Island.

In [ ]:
#I will do the first one. We want flows from Manhattan to anywhere OUTSIDE New York state--i.e., destination GEOIDs that do NOT start with '36'.
flows_out_of_Manhattan_to_outside_NY = flows_out_of_Manhattan.loc[~flows_out_of_Manhattan['Destination'].str.startswith('36')].groupby('Origin')['Flow'].sum()

#We do the .groupby('Origin') because we want to sum all flows from each Manhattan CBGs to outside NY state.
# There are multiple destinations for each origin, so we need to group by origin and sum the flows. You could play around with mapping by destination too.
# In general, make sure to add this groupby statement after the following lines:

#Flows to NY state:
flows_out_of_Manhattan_to_NY = flows_out_of_Manhattan.loc[...].groupby('Origin')['Flow'].sum()

#Flows to NY state BUT outside NYC---we will use and & (and) operator:
flows_out_of_Manhattan_to_NY = flows_out_of_Manhattan.loc[(<here use the query for flows to NY state, as before>)&(<here use the query for flows NOT in NYC>)].groupby('Origin')['Flow'].sum()
#Hint: str.startswith allows you to supply a tuple of strings to test for i.e. .str.startswith(('36061', '36005',...,))


#And, very importantly, we need the stayers---select rows where origin = destination:
flows_to_the_same_CBG = flows_out_of_Manhattan[...].groupby('Origin')['Flow'].sum()

In [ ]:
#I will do the first one. We want flows from Manhattan to anywhere OUTSIDE New York state--i.e., destination GEOIDs that do NOT start with '36'.
flows_out_of_Manhattan_to_outside_NY = flows_out_of_Manhattan.loc[~flows_out_of_Manhattan['Destination'].str.startswith('36')].groupby('Origin')['Flow'].sum()

#We do the .groupby('Origin') because we want to sum all flows from each Manhattan CBGs to outside NY state.
# There are multiple destinations for each origin, so we need to group by origin and sum the flows. You could play around with mapping by destination too.
# In general, make sure to add this groupby statement after the following lines:

#Flows to NY state:
flows_out_of_Manhattan_to_NY = flows_out_of_Manhattan.loc[...].groupby('Origin')['Flow'].sum()

#Flows to NYC:
flows_out_of_Manhattan_to_NYC = flows_out_of_Manhattan.loc[...].groupby('Origin')['Flow'].sum()
#Hint: str.startswith allows you to supply a tuple of strings to test for i.e. .str.startswith(('36061', '36005',...,))

#Flows to outside of NYC:
flows_out_of_Manhattan_to_outside_NYC = flows_out_of_Manhattan.loc[...].groupby('Origin')['Flow'].sum()
#Hint: you can use ~ for negation

#Flows to outside of Manhattan, but still in NYC:
flows_out_of_Manhattan_to_outside_Manhattan = flows_out_of_Manhattan.loc[...].groupby('Origin')['Flow'].sum()

#And, very importantly, we need the stayers---select rows where origin = destination:
flows_to_the_same_CBG = flows_out_of_Manhattan[...].groupby('Origin')['Flow'].sum()

#And also the total flows from each CBG:
total_flows = flows_out_of_Manhattan.groupby('Origin')['Flow'].sum()

#To get the total movers, we need to subtract the stayers from the total flows:
total_movers = total_flows - flows_to_the_same_CBG

Note that for many of the above quantities, we will want to also subtract the `flows_to_the_same_CBG` --- stayer entries are very large, and it makes more sense to express quantities as shares of movers unless the goal is to show striking migration rates. In some cases, however, you won't need to do that---for example, the flows to outside ny state already exclude folks who stay in the same CBG:

In [ ]:
total_movers = total_movers
movers_to_NYC = flows_out_of_Manhattan_to_NYC - flows_to_the_same_CBG #here we need to discount!
movers_to_outside_NYC = flows_out_of_Manhattan_to_outside_NYC
movers_to_outside_Manhattan = flows_out_of_Manhattan_to_outside_Manhattan

Now let's include some quantities in the GeoDataFrame. We will merge the series above on the GEOID10 column

In [ ]:
#First, select Manhattan CBGs from the NYC geodataframe:
Manhattan_CBGs = NYC_CBGs_nowater[NYC_CBGs_nowater['COUNTYFP10']=='061'].reset_index(drop=True)
Manhattan_CBGs['GEOID10'] = Manhattan_CBGs['GEOID10'].astype(str).str.zfill(12) #Make sure GEOID10 is a string of length 12


#First, include the number of total movers:
Manhattan_CBGs = Manhattan_CBGs.merge(total_movers.rename('Total Movers'), how='left', left_on='GEOID10', right_index=True)

#Use these columns names: ['Movers to NYC', 'Movers to outside NYC', 'Movers to outside Manhattan but still NYC']

#Include movers to NYC:
Manhattan_CBGs = ...

#Include movers to outside NYC:
Manhattan_CBGs = ...

#Include movers to outside Manhattan:
Manhattan_CBGs = ...

Normalize the columns dividing by the number of movers:

In [ ]:
for column in ['Movers to NYC', 'Movers to outside NYC', 'Movers to outside Manhattan but still NYC']:
    Manhattan_CBGs[f'{column} (fraction)'] = Manhattan_CBGs[column] / Manhattan_CBGs['Total Movers']

Now the moment we have all been waiting for!

In [ ]:
fig, Axes = plt.subplots(ncols=3, figsize=(25,10))
for ax_idx, column in enumerate(['Movers to NYC', 'Movers to outside NYC', 'Movers to outside Manhattan but still NYC']):
    _ = Manhattan_CBGs.plot(column=f'{column} (fraction)', ax=Axes[ax_idx], legend=True, cmap='YlOrBr', missing_kwds={'color':'lightgrey', 'label':'No data'})
    _ = Axes[ax_idx].set_title(column, fontsize=15)
    _ = Axes[ax_idx].axis('off')

Note that the plot isn't quite illuminating at first---this is normal! We are seeing noise picked up much strongerly (so places like Central Park and Randall's Island have a high share, likely due to their very small number of people). As we are comparing three plots, we would also like to have them on the same scale. We can define `vmin` and `vmax` in our plot to standardize the bar, we can change the colorscale to something that allows for more nuances, and we can also omit CBGs with less than 5 movers.

In [ ]:
fig, Axes = plt.subplots(ncols=3, figsize=(25,10))
for ax_idx, column in enumerate(['Movers to NYC', 'Movers to outside NYC', 'Movers to outside Manhattan but still NYC']):
    _ = Manhattan_CBGs.plot(column=f'{column} (fraction)', ax=Axes[ax_idx], legend=True, cmap='YlOrBr', missing_kwds={'color':'lightgrey', 'label':'No data'},
                            vmin=0, vmax=1., legend_kwds={'extend': 'neither'})#'max'})  # extend above vmax, crucial if you are setting vmax below 1! use 'neither', 'max', 'both', 'min'
    #Plot, in grey, the CBGs with less than 5 movers:
    _ = Manhattan_CBGs[Manhattan_CBGs['Total Movers'] <= 5].plot(...#color='lightgrey', ax=Axes[ax_idx], legend=False)
            
    _ = Axes[ax_idx].set_title(column, fontsize=15)
    _ = Axes[ax_idx].axis('off')

Ok! Now we see that the plots all have very different ranges---what stories can you tell about the geographical disparities you see? To tell another story, I would suggest that you restrict vmax to something where you can see color variation on the THIRD map---people who move outside of Manhattan but stay in New York City. What can be a socioeconomical reason for the results you are seeing?

## 3. Mapping Out-Migration from Wildfires

Now let's look at migration in CBGs affected by the Camp fire in California---you can use this as an opportunity to learn about spatial joins! The wildfire [data,](https://www.fire.ca.gov/what-we-do/fire-resource-assessment-program/fire-perimeters) comes from the California department of Forestry and Fire Protection. I included the file in the `data` directory:

In [ ]:
wildfire_perimeter = gpd.read_file('data/wildfire_polygons.gdb', layer='firep24_1') #Read a geodatabase file by reading the folder!

Check how the dataframe looks like:

In [ ]:
wildfire_perimeter.head()

Let's select the rows corresponding to CAMP fire. note that simply selecting by name may be insufficient---you might need to use the YEAR or other data to select the correct row:

In [ ]:
camp_fire_gdf = wildfire_perimeter[...] #Select the Camp Fire, by name
display(camp_fire_gdf)
#HINT: use & for "and" conditions, | for "or" conditions. You may need to know the month and year of camp fire. Do one query, print all rows, and refine it until you get the right one.
# Your final query should return exactly one row.

Can you plot this perimeter?

In [ ]:
#Some interesting arguments of GeoDataFrame.plot include:
# - color to define a single color for all geometries
# - edgecolor to define the color of the geometry borders
# - linewidth to define the width of the geometry borders
# - alpha to define transparency (0=fully transparent, 1=fully opaque)
# You can also call .boundary on a GeoSeries or GeoDataFrame to get just the boundaries of the geometries, which you can then plot.

fig, ax = plt.subplots(figsize=(10,10))
_ = camp_fire_gdf.plot(color='maroon', alpha=0.8, ax=ax)
_ = ax.set_title('Camp Fire Perimeter', fontsize=15)

### Geographic projections:

<b> This might be the most important learning goal of the workshop: </b> Spatial data has <i>projections</i>. The Eart is round, and representing it in a two-dimensional way requires that we define a coordinate reference system (crs). For example, you may have learned about Mercator, Peters, and other cartographical projections. You can query your GeoDataFrame crs by using the `crs` attribute:

In [ ]:
wildfire_perimeter.crs

Importantly, look at the dimensions: this is a metric projection in meters, so any length or area operations will give results in meters. Now look at the crs of the California CBG polygons:

In [ ]:
CA_CBGs.crs

This is a <b> geodetic </b> projection, meaning that it simply flattens the Earth's sphere respecting latitude and longitude---it is highly interpretable (you can just look at lat long and query, for example), but it will not work for any geographic operations (e.g. computing lengths, or areas, or joining things because this requires computing distances). For example, look at the warnings you receive:

In [ ]:
CA_CBGs.iloc[:10].area

It's good practice to always keep a variable with your crs around. For this case, let's project the CA_CBGs file to the same crs as the wildfire polygons (which is in meters):

In [ ]:
CA_CBGs_projected = CA_CBGs.to_crs(wildfire_perimeter.crs)

Try computing the area now---what are these units?

In [ ]:
CA_CBGs_projected.iloc[:10].area

### Joining Spatial Data

We have a GeoDataFrame of a wildfire polygon and a GeoDataFrame of Census Block Groups. Let's ask ourselves the question of which CBGs are affected by the fire. First, let's visualize the CBGs and the fire polygon:

In [ ]:
#Plot both the fire perimeter and the CBGs in the same axis. If they are on the same projection, they should line up.
fig, ax = plt.subplots(figsize=(10,10))
#YOUR PLOTTING CODE HERE

You can zoom into the fire perimeter either by seeting ax limits (use `ax.set_xlim` and `ax.set_ylim`) or by using `cx` on a GeoDataFrame. The latter is more useful if you want to avoid plotting some CBGs, but it will show things outside the polygon if they at some point intersect the polygon:

In [ ]:
total_bounds = camp_fire_gdf.total_bounds  # returns (minx, miny, maxx, maxy)
cropped_CBGs = CA_CBGs_projected.cx[total_bounds[0]:total_bounds[2], total_bounds[1]:total_bounds[3]] #this is a square-like box cropping!

#Now plot the cropped CBGs with the same code you wrote above:
fig, ax = plt.subplots(figsize=(10,10))
#YOUR PLOTTING CODE HERE

To actually selec the CBGs intersecting the polygon, we can use `overlay`

In [ ]:
CA_CBGs_projected.overlay()

Select the CBGs. Use the `how='intersection'` parameter:

In [ ]:
CA_CBGs_in_Camp = CA_CBGs_projected.overlay(...)

Plot the `CA_CBGs_in_Camp` dataset:

In [ ]:
CA_CBGs_in_Camp.plot()

One caveat: this cropped some CBGs, showing all those partially affected by the fire. One important consideration is the fraction of each CBG affected. To understand that, you will need to compute area fractions.

In [ ]:
#Compute the area of each CBG in the cropped dataset:
area_in_camp = CA_CBGs_in_Camp.set_index('GEOID10').area #setting the index to GEOID10 makes it easier to look up areas by GEOID later

#Compute the area of each CBG in California---remember to read about projections above!:
area_in_CA = ...

#Compute the area fraction of each CBG affected by the fire:
# You might want to fill nans with zeroes.
area_fraction_in_camp = ...

Merge the `area_fraction_in_camp` series on the `GEOID10` column back into the `CA_CBGs_projected` geodataframe, then plot. You can set vmin and vmax to 0 and 1.

In [ ]:
#Merge the area fractions back into the original CA_CBGs_projected geodataframe:
CA_CBGs_projected = CA_CBGs_projected.merge(area_fraction_in_camp...) #Do similar to how we merged in the Manhattan example above
CA_CBGs_projected.plot('Fraction in Camp Fire'...)

Try to also zoom:

In [ ]:
fig, ax = plt.subplots(figsize=(10,10))
#YOUR PLOTTING CODE HERE

#Zoom with limits:
_= ax.set_xlim(total_bounds[0], total_bounds[2])
_ = ax.set_ylim(total_bounds[1], total_bounds[3])

For some analysis, you might want to set a threshold and only consider CBGs with more than $\alpha \%$ of their area inside a polygon, for example.

### Visualizing out-migration

Now let's visualize out-migration from MIGRATE. Let's first select flows out of all CBGs within $25km$ from the fire perimeter:

In [ ]:
#To find CBGs within a certain distance from the fire perimeter, we can use the .buffer() method to create a buffer around the fire perimeter, then use overlay
buffer_distance = 25*1000 #25km in meters (this is the unit of the projection!)
buffered_polygon = camp_fire_gdf.geometry.buffer(buffer_distance) #This creates as GeoSeries! You will need to create a GeoDataFrame if you want to use overlay.
buffered_polygon_gdf = gpd.GeoDataFrame(geometry=buffered_polygon)

#Use overlay:
CA_CBGs_of_interest = CA_CBGs_projected.overlay(...)

#Plot to visaulize (along with fire polygon)---I recommend setting linewidth=1 and edgecolor='black' for the CBGs gdf to see CBG boundaries:
fig, ax = plt.subplots(figsize=(10,10))
#YOUR PLOTTING CODE HERE

Now select from MIGRATE_df all outflows from those CBGs---the same thing you did for New York. Note that now you might want to first query what CBGs have GEOIDs in your selected gdf, and then query for rows of `MIGRATE_df` with those GEOIDs in `Origin`:

In [ ]:
MIGRATE_data['Origin'] = MIGRATE_data['Origin'].astype(str).str.zfill(12) #this might already be done!
MIGRATE_data['Destination'] = MIGRATE_data['Destination'].astype(str).str.zfill(12) #this might already be done!

#Get a list of GEOIDs of CBGs of interest using .values on the geodataframe above and query MIGRATE_data for outflows from those CBGs-:
GEOIDs_to_query = ...
selected_outflows = MIGRATE_data.loc[...]

#Remember that we would like to aggregate by origin:
total_outflows = ...

#Also, get stayers and the total movers, as you did for Manhattan above:
flows_to_the_same_CBG = ...
total_movers = ...

Visualize the fraction of flows who moved out of the CBG (this is the out-migration rate, as the total `flow` in our terminology is people who moved out + people who stayed)

In [ ]:
#Compute the outmigration rate:
outmigration_rate = ...

#Merge into the GeoDataFrame:
CA_CBGs_of_interest = CA_CBGs_of_interest.merge(...)

#Plot:
fig, ax = plt.subplots(figsize=(10,10))
#your plotting code here

## 4. More Fun Stuff if you have time!

If you got here, it means you are a pro at plotting and visualizing spatial data already! Here are some interesting analyses you could play with:

- Can you figure out destinations of folks moving from CBGs affected by Camp Fire? How many people (in share of movers) end up in CBGs inside the fire perimeter? How many people end up in an affected county---you might want to look up counties---or an adjacent county but outside the fire perimeter? How many people leave California altogether?
- What about cross-area flows. You have access to all flows departing from New York and California. How many people are moving from each NYC borough to San Francisco---this is basically a county-to-county flow, which we now can verify yearly but beforehand we only had access to at a 5-year aggregate. How many people from each NYC CBG are doing that? 